# Dong Boundary Condition: Steady-State Convergence study computing Kovasznay Flow

First test problem for validating the outflow B.C. by S. Dong. The exact solution for the velocity and pressure fields of the Kovasznay flow is given by:

>$$  \begin{align*} 
u &= 1 - \textrm{e}^{\lambda x} \cos{(2 \pi y)}, \\
v &= \frac{\lambda}{2 \pi} \textrm{e}^{\lambda x} \sin{(2 \pi y)}, \\
p &= \frac{1}{2} (1 - \textrm{e}^{2 \lambda x})
\end{align*}$$

where $\lambda = \frac{1}{2 \nu} - \sqrt{\frac{1}{4 \nu^2} + 4 \pi^2}$ with $\nu = \frac{1}{40}$.

In [ ]:
//#r "../../src/L4-application/BoSSSpad/bin/Release/net5.0/BoSSSpad.dll"
//#r "../../src/L4-application/BoSSSpad/bin/Debug/net5.0/BoSSSpad.dll"
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using BoSSS.Solution.NSECommon;

In [ ]:
var myBatch = ExecutionQueues[1];
myBatch

RuntimeLocation,AdditionalEnvironmentVars,DeploymentBaseDirectory,DeployRuntime,Name,DotnetRuntime,Username,NumOfAdditionalServiceCores,NumOfAdditionalServiceCoresMPISerial,NumOfServiceCoresPerMPIprocess,ServerName,ComputeNodes,DefaultJobPriority,SingleNode,AllowedDatabasesPaths
win\amd64,[ ],\\dc3\userspace\smuda\hpccluster\binaries,True,FDY-WindowsHPC,dotnet,FDY\smuda,0,0,0,DC3,<null>,Normal,True,[ \\dc3\userspace\smuda\hpccluster\ ]


In [ ]:
BoSSSshell.WorkflowMgm.Init("KovasznayFlow", myBatch);

## Grid generation

In [ ]:
int[] Resolutions = new int[] { 4 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"KovasznayFlow_{Res}x{2*Res}_DongBC";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    //cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(-0.5, -0.1, Res + 1);
        double[] yNodes = GenericBlas.Linspace(-0.5, 0.5, (2 * Res) + 1);
        
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes, periodicY:true);
        grd.Name = GridName;
        
        grd.DefineEdgeTags(delegate(Vector X) {
            string ret = null;
            if((X.x + 0.5).Abs() <= 1e-8)
                ret = IncompressibleBcType.Velocity_Inlet.ToString();
            if((X.x + 0.1).Abs() <= 1e-8)
                ret = IncompressibleBcType.Dong_OutFlow.ToString();
            return ret;
        });        
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values